# Report + Business Stakeholder Review — Router Review

Human-run notebook for the **router slice** of Sean Step 6: the new `route_after_business_review` function in `orchestration/router.py`.

Plan: `project_planning/sean_step_artifacts/Report_Business_Review_Implementation_Plan.md`  
Checklist: `project_planning/sean_step_artifacts/Report_Business_Review_Checklist.md`

Run cells top to bottom to:
- inspect the router function and its three sinks (accept / revise_report / revise_modeling)
- confirm the iteration cap reads from `config/workflows.yaml`
- exercise every routing branch with a hand-built state
- confirm the cap forces an `end` once the loop budget is spent

In [ ]:
from pathlib import Path
import inspect
import subprocess
from pprint import pprint

from multi_agent_ds.orchestration.router import (
    route_after_business_review,
    _modeling_iteration_limit,
    _report_iteration_limit,
)
from multi_agent_ds.core import load_workflows_config

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root.")

ROOT = resolve_repo_root()
print("Repo root:", ROOT)

def run_pytest(args: list[str]) -> None:
    cmd = ["uv", "run", "pytest", *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"pytest failed with exit code {completed.returncode}")


## 1. Inspect the router function

**Human review questions:**
- Does the function read `business_review.next_action` and only that?
- Are the three sinks named correctly (`end`, `report_writer`, `ml_modeler_baseline`)?
- Are the iteration caps reading from config rather than being hard-coded?

In [ ]:
print(inspect.getsource(route_after_business_review))


## 2. Confirm the iteration caps come from config

`config/workflows.yaml` has separate `modeling.max_iterations` and `report.max_iterations` entries. The router reads them via the helpers below.

In [ ]:
wf = load_workflows_config()["workflows"]
print("Modeling cap (config):", wf["modeling"]["max_iterations"])
print("Report cap   (config):", wf["report"]["max_iterations"])
print("Modeling cap (resolved):", _modeling_iteration_limit())
print("Report cap   (resolved):", _report_iteration_limit())


## 3. Branch matrix

Walk through every branch of the router with a hand-built state. The cell prints which next-node the router would emit.

In [ ]:
def show(case: str, state: dict) -> None:
    print(f"{case:50s} -> {route_after_business_review(state)!r}")

# Accept
show("accept verdict", {"business_review": {"next_action": "accept"}})

# revise_report under cap
show(
    "revise_report, report_iteration=1 (under cap)",
    {"business_review": {"next_action": "revise_report"}, "report_iteration": 1},
)

# revise_report at cap
show(
    f"revise_report, report_iteration={_report_iteration_limit()} (at cap)",
    {"business_review": {"next_action": "revise_report"}, "report_iteration": _report_iteration_limit()},
)

# revise_modeling under cap
show(
    "revise_modeling, modeling_iteration=1 (under cap)",
    {"business_review": {"next_action": "revise_modeling"}, "modeling_iteration": 1},
)

# revise_modeling at cap
show(
    f"revise_modeling, modeling_iteration={_modeling_iteration_limit()} (at cap)",
    {"business_review": {"next_action": "revise_modeling"}, "modeling_iteration": _modeling_iteration_limit()},
)

# Defensive: missing review
show("missing business_review", {})


## 4. Run the router tests

Six focused tests covering accept, both revise paths under their caps, both force-accepts at cap, and the missing-review default.

In [ ]:
run_pytest([
    "tests/test_report_business_review.py",
    "-k",
    "route_after_business_review",
    "-v",
])


## 5. Sign-off

If the router routes correctly to every sink and respects both iteration caps, tick the **Step 4 (router)** human-review boxes in `Report_Business_Review_Checklist.md`.